In [5]:
import sys
sys.path.append("../../")

import numpy as np

#
# TO BE USED WITH godot/mecanum_4w
#

from lib.system.mecanum import *
from lib.dds.dds import *
from lib.utils.time import *
from lib.system.controllers import *
from lib.data.dataplot import *

class MecanumWheelsController:
    
    def __init__(self, _kp, _ki, _sat):
        self.controllers = []
        self.torques = [0,0,0,0]
        for i in range(4):
            self.controllers.append(PID_Controller(_kp, _ki, 0, _sat))
    
    def evaluate(self, delta_t, w_target, w_current):
        for i in range(4):
            self.torques[i] = self.controllers[i].evaluate(delta_t, w_target[i] - w_current[i])
        return self.torques

dds = DDS()
dds.start()

dds.subscribe(['tick'])

cart = MecanumFourWheels(10, # mass, 10kg 
                         0.7, # wheelbase
                         0.8, # lin-friction 
                         0.9, # ang-friction 
                         0.03) # 3cm radius motion wheel 
cart.set_pose(0,0,math.radians(30))

wheel_controllers = MecanumWheelsController(0.05, 0.01, 5.0)

vx_local_target = 0.0
vy_local_target = 0.1
w_local_target = 0

v_local_targets = np.array( [vx_local_target, vy_local_target, w_local_target])

t = Time()
t.start()
while t.get() < 5:

    #dds.wait('tick')
    t.sleep(0.001)
    delta_t = t.elapsed()
    
    w_targets = cart.ik_matrix @ v_local_targets
    w = cart.get_wheel_speed()
    torque = wheel_controllers.evaluate(delta_t, w_targets.flatten().tolist(), w)

    cart.evaluate(delta_t, torque[0], torque[1], torque[2], torque[3])
    pose = cart.get_pose()

    dds.publish('X', pose[0], DDS.DDS_TYPE_FLOAT)
    dds.publish('Y', pose[1], DDS.DDS_TYPE_FLOAT)
    dds.publish('Theta', pose[2], DDS.DDS_TYPE_FLOAT)
    
    dds.publish('w1', w[0], DDS.DDS_TYPE_FLOAT)
    dds.publish('w2', w[1], DDS.DDS_TYPE_FLOAT)
    dds.publish('w3', w[2], DDS.DDS_TYPE_FLOAT)
    dds.publish('w4', w[3], DDS.DDS_TYPE_FLOAT)

dds.publish('w1', 0, DDS.DDS_TYPE_FLOAT)
dds.publish('w2', 0, DDS.DDS_TYPE_FLOAT)
dds.publish('w3', 0, DDS.DDS_TYPE_FLOAT)
dds.publish('w4', 0, DDS.DDS_TYPE_FLOAT)
dds.stop()

